# Parte 2b: Modelo B (preentrenado, InsightFace `buffalo_l`)

> **Notebook auxiliar.** Se usa para trabajar en paralelo. Su contenido se integra en `2-train.ipynb`, que es la que se entrega.
> La lógica va en `src/lib/`; acá solo se llaman funciones. Al terminar, guarda los embeddings con `save_embeddings` (contrato en `src/lib/evaluation/embedding_store.py`).

In [ ]:
import json
import sys
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from lib.evaluation import EmbeddingSet, load_embeddings, results_path, save_embeddings

CONFIG = json.loads((ROOT / "splits" / "config.json").read_text(encoding="utf-8"))
ALIGNED_ROOT = ROOT / CONFIG["aligned_root"]
SEED = CONFIG["seed"]
splits = pd.read_csv(ROOT / "splits" / "splits.csv")
splits["split"].value_counts()

In [ ]:
def load_faces(rows: pd.DataFrame) -> list[np.ndarray]:
    """Caras alineadas en BGR, tal como las guardó la Parte 1."""
    return [cv2.imread(str(ALIGNED_ROOT / p)) for p in rows["aligned_path"]]

## 1. Descripción del modelo
Completar y verificar contra la documentación de InsightFace:
- **Detector:** SCRFD `det_10g.onnx` (el mismo de la Parte 1).
- **Reconocedor:** `w600k_r50.onnx`: arquitectura, datos de entrenamiento y pérdida.
- **Dimensión del embedding:** 512.
- **Licencia:** modelos preentrenados de InsightFace solo para uso no comercial / investigación.

## 2. Extracción de embeddings
Sin reentrenar. Usa el mismo `OnnxFaceEmbedder` y la misma normalización que la app.

In [ ]:
from lib.vision import OnnxFaceEmbedder

embedder = OnnxFaceEmbedder.from_model_pack("buffalo_l")
eval_rows = splits[splits["split"] != "train"].reset_index(drop=True)
embeddings = embedder.embed(load_faces(eval_rows))
embeddings.shape

In [ ]:
save_embeddings(
    EmbeddingSet(
        model="insightface_buffalo_l",
        paths=eval_rows["aligned_path"].to_numpy(),
        labels=eval_rows["identity"].to_numpy(),
        splits=eval_rows["split"].to_numpy(),
        embeddings=embeddings,
    ),
    results_path(ROOT, "insightface_buffalo_l"),
)

## 3. Evaluación
Las funciones de 3.3 (verificación, identificación, umbral, clustering, errores, costo) van en `src/lib/evaluation/` y se prueban acá sobre estos embeddings. Después se llaman igual desde `2-train.ipynb` para ambos modelos.

In [ ]:
# TODO: probar las funciones de lib.evaluation sobre load_embeddings(results_path(ROOT, "insightface_buffalo_l"))